# 01 - Deploy Foundry and the public LiteLLM proxy

This notebook creates the Foundry model, builds the LiteLLM image in ACR, and deploys an external HTTPS Container App. The proxy is disabled during the first Bicep pass so the Foundry API key can be retrieved and injected as a secret during the second pass.

**Before deploying:** validate `MODEL_NAME` and `MODEL_VERSION` in the selected region. GPT-5.4 availability and version strings can change.

In [ ]:
import json
import os
import secrets
import subprocess
import sys
from pathlib import Path

RESOURCE_GROUP = 'rg-litellm01-foundry-claude'
LOCATION = 'eastus2'
DEMO_ID = 'litellm01'
MODEL_NAME = 'gpt-5.4'
MODEL_VERSION = '2026-03-05'
DEPLOYMENT_NAME = 'gpt-5.4'
DEPLOYMENT_NAME_ALIAS = 'azure-gpt-5.4'
DEPLOYMENT_NAME_FOR_BICEP = DEPLOYMENT_NAME

demo_root = Path('..').resolve()
if str(demo_root) not in sys.path:
    sys.path.insert(0, str(demo_root))
from app.notebook_helpers import resolve_az_cli
az_cmd = resolve_az_cli()
if not az_cmd:
    raise RuntimeError('Azure CLI not found. Run 00_setup.ipynb and ensure az is on PATH.')
print(f'Using Azure CLI: {az_cmd}')
print(f'Resource group: {RESOURCE_GROUP}')
print(f'Foundry model: {MODEL_NAME} / {MODEL_VERSION}')

## Step 1 - Deploy the Foundry account, model, registry, and Container Apps environment

In [ ]:
def run_az(args, allow_failure=False):
    result = subprocess.run([az_cmd, *args], capture_output=True, text=True)
    if result.returncode != 0 and not allow_failure:
        raise RuntimeError('az ' + ' '.join(args) + ' failed (exit ' + str(result.returncode) + ')\n' + 'stderr: ' + result.stderr.strip() + '\n' + 'stdout: ' + result.stdout.strip())
    return result

run_az(['group', 'create', '--name', RESOURCE_GROUP, '--location', LOCATION, '--tags', 'SecurityControl=Ignore', '-o', 'none'])
run_az([
    'deployment', 'group', 'create', '--resource-group', RESOURCE_GROUP,
    '--name', 'litellm01-bootstrap', '--template-file', '../infra/main.bicep',
    '--parameters', f'location={LOCATION}', f'demoId={DEMO_ID}',
    f'deploymentName={DEPLOYMENT_NAME_FOR_BICEP}', f'modelName={MODEL_NAME}',
    f'modelVersion={MODEL_VERSION}'
])
print('Bootstrap deployment completed. The public proxy is not enabled yet.')

## Step 2 - Retrieve the Foundry API key and build the LiteLLM image

In [ ]:
outputs = json.loads(run_az(['deployment', 'group', 'show', '-g', RESOURCE_GROUP, '-n', 'litellm01-bootstrap', '--query', 'properties.outputs', '-o', 'json']).stdout)
def output(name):
    return (outputs.get(name) or {}).get('value', '')
ai_account_name = output('aiServicesAccountName')
acr_name = output('acrName')
acr_login_server = output('acrLoginServer')
ai_endpoint = output('aiServicesEndpoint')
if not ai_account_name or not acr_name:
    raise RuntimeError(f'Expected deployment outputs were missing: {list(outputs)}')

api_key_result = run_az(['cognitiveservices', 'account', 'keys', 'list', '--resource-group', RESOURCE_GROUP, '--name', ai_account_name, '--query', 'key1', '-o', 'tsv'])
azure_api_key = api_key_result.stdout.strip()
if not azure_api_key:
    raise RuntimeError('Foundry account key was empty. Confirm local auth is enabled on the AIServices account.')

image_name = 'litellm-proxy'
image_tag = 'latest'
image_reference = f'{acr_login_server}/{image_name}:{image_tag}'
build = run_az(['acr', 'build', '--registry', acr_name, '--image', f'{image_name}:{image_tag}', '../app'])
print(build.stdout)
print(f'Image built: {image_reference}')

## Step 3 - Enable the public proxy and write the connection details

In [ ]:
proxy_master_key = 'sk-' + secrets.token_urlsafe(32)
run_az([
    'deployment', 'group', 'create', '--resource-group', RESOURCE_GROUP,
    '--name', 'litellm01-proxy', '--template-file', '../infra/main.bicep',
    '--parameters', f'location={LOCATION}', f'demoId={DEMO_ID}',
    f'deploymentName={DEPLOYMENT_NAME_FOR_BICEP}', f'modelName={MODEL_NAME}',
    f'modelVersion={MODEL_VERSION}', f'containerImageReference={image_reference}',
    f'azureApiKey={azure_api_key}', f'proxyMasterKey={proxy_master_key}'
])
proxy_outputs = json.loads(run_az(['deployment', 'group', 'show', '-g', RESOURCE_GROUP, '-n', 'litellm01-proxy', '--query', 'properties.outputs', '-o', 'json']).stdout)
proxy_url = (proxy_outputs.get('proxyUrl') or {}).get('value', '')
if not proxy_url:
    raise RuntimeError('Proxy URL was empty. Inspect the litellm01-proxy deployment operations.')

env_path = demo_root / 'env' / '.env'
env_path.parent.mkdir(parents=True, exist_ok=True)
env_path.write_text('\n'.join([
    f'AZURE_SUBSCRIPTION_ID={run_az(["account", "show", "--query", "id", "-o", "tsv"]).stdout.strip()}',
    f'AZURE_RESOURCE_GROUP={RESOURCE_GROUP}',
    f'AZURE_AI_ENDPOINT={ai_endpoint}',
    f'AZURE_AI_ACCOUNT_NAME={ai_account_name}',
    f'AZURE_MODEL_DEPLOYMENT={DEPLOYMENT_NAME_FOR_BICEP}',
    f'LITELLM_MODEL_ALIAS={DEPLOYMENT_NAME_ALIAS}',
    f'LITELLM_PROXY_URL={proxy_url}',
    f'ANTHROPIC_BASE_URL={proxy_url}',
    f'LITELLM_MASTER_KEY={proxy_master_key}',
    f'ANTHROPIC_AUTH_TOKEN={proxy_master_key}',
    'CLAUDE_CODE_ENABLE_GATEWAY_MODEL_DISCOVERY=1',
    f'CLAUDE_CODE_MODEL={DEPLOYMENT_NAME_ALIAS}',
    f'AZURE_API_KEY={azure_api_key}',
]) + '\n', encoding='utf-8')
print(f'Public LiteLLM URL: {proxy_url}')
print(f'Claude Code model: {DEPLOYMENT_NAME_ALIAS}')
print(f'API key: {proxy_master_key[:8]}... (saved to env/.env)')
print(f'Connection details written to {env_path}')